In [52]:
import numpy as np
import functions as fn
from circuit_obj import Circuit
from tqdm import tqdm
from itertools import product
from matplotlib import pyplot as plt
import json

In [73]:
# Define your parameter arrays
N = 14
T = 1000
L = 3 # How fine the grid: L x L
circuit_realizations = 10

masks_dict = fn.load_mask_memory(N, 2)

In [74]:
def initial_state(N, n):
    """
    Generate a random state vector on N qubits with spin up at site n
    """
    up = np.array([0, 1])
    if n == 0:
        rnd_part = np.random.rand(2**(N - 1)) + 1j * np.random.rand(2**(N - 1))
        state = np.kron(up, rnd_part)
        state /= np.linalg.norm(state)
        return state
    elif n == N - 1:
        rnd_part = np.random.rand(2**(N - 1)) + 1j * np.random.rand(2**(N - 1))
        state = np.kron(rnd_part, up)
        state /= np.linalg.norm(state)
        return state
    part1 = np.random.rand(2**(n)) + 1j * np.random.rand(2**(n))
    part2 = np.random.rand(2**(N - n - 1)) + 1j * np.random.rand(2**(N - n - 1))
    state = np.kron(part1, np.kron(up, part2))
    state /= np.linalg.norm(state)
    return state

# sanity check :
# [np.round(fn.get_magnetization(initial_state(N, idx), N)[idx]) for idx in range(N)]

st = initial_state(N, 0)

In [75]:

with open(f'../random_archive/angles_N{N}.json', 'r', encoding='utf-8') as f:
    loaded_dict = json.load(f)
    
circuits = []

for cr in range(circuit_realizations):
    circuits_Jx = []
    for Jx in np.linspace(.1, np.pi-.1, L):
        circuits_Jz = []
        for Jz in np.linspace(.1, np.pi-.1, L):
            gates = []
            for n in range(N):
                params = loaded_dict[cr][n]
                # Jz = params['Jz']; Jx = params['Jx']
                θ1 = params['θ1']; θ2 = params['θ2']
                θ3 = params['θ3']; θ4 = params['θ4']
                gates.append(fn.gate_xyz_disordered(θ1, θ2, Jx/4, -Jx/4, Jz/4, θ3, θ4)) # h1, h2, Jx, Jy, Jz, h3, h4
            
            gates = np.array(gates)
            assert len(gates) == N, f"Expected {N} gates, got {gates.shape}"

            order = fn.gen_gates_order(N)    
            circuit = Circuit(N=N, gates=gates, order=order)
            circuit.couplings = [Jx, Jz]
            circuit.verbose = False
            circuits_Jz.append(circuit)
        circuits_Jx.append(circuits_Jz)
    circuits.append(circuits_Jx)
            
def random_basis_state(n_qubits, seed=None):
    if seed is not None:
        np.random.seed(seed)    
    x = np.random.randint(0, 2**n_qubits)
    psi = np.zeros(2**n_qubits, dtype=complex)
    psi[x] = 1.0
    return psi

##############################################################################################

def compute_correlation(idx, circuit):
    state = initial_state(N, idx) # initial_state_test(theta)
    return circuit.run(masks_dict, state, T, objective=['correlation'])

def compute_magic(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['magic'])

def compute_entanglement(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['entanglement'])

def compute_magic_entanglement(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['magic', 'entanglement'])

In [ ]:
if globals().get('correlation') is None or globals(
    ).get('correlation').shape != (N, circuit_realizations * (L**2), T + 1, N):
    correlation = np.zeros((N, circuit_realizations * (L**2), T + 1, N), dtype=np.float64)
    
if globals().get('magic') is None or globals(
    ).get('magic').shape != (circuit_realizations, L, L, T + 1):
    magic = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)
    
if globals().get('entan') is None or globals(
    ).get('entan').shape != (circuit_realizations, L, L, T + 1):
    entan = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)

for idx, Lx, Lz, cr in tqdm(
        product(
            range(1),
            range(L),
            range(L),
            range(circuit_realizations),
        ),
        total=1 * L * L * circuit_realizations,
        desc='Computing magic and entanglement'):
    circuit = circuits[cr][Lx][Lz]
    output, _ = compute_magic_entanglement(circuit);
    magic[cr, Lx, Lz, :], entan[cr, Lx, Lz, :] = output[0], output[1]

Computing magic and entanglement:  89%|████████▉ | 80/90 [22:39<03:09, 18.92s/it]

In [ ]:
# # np.save('result_N16.npy', result)
# result = np.load('result_N16.npy')
# with open("nathan.txt", "r") as f:
#     lines = f.readlines()                    # read all lines as strings
# data =[float(i) for i in lines[0].split(', ')]  # drop blank lines
# print(data)

In [ ]:
%matplotlib osx

fig, axs = plt.subplots(L, L, figsize=(12, 10), sharex=True, sharey=True)
for Lx, Lz in product(range(L), range(L)):
    Jx = np.linspace(.1, np.pi-.1, L)[Lx]
    Jz = np.linspace(.1, np.pi-.1, L)[Lz]
    axs[L-Lx-1, Lz].plot(magic[:,Lx,Lz,:].mean(axis=(0)), label=f'Magn Jx={Jx:.2f}π, Jz={Jz:.2f}π')
    axs[L-Lx-1, Lz].plot(entan[:,Lx,Lz,:].mean(axis=(0)), label=f'Ent Jx={Jx:.2f}π, Jz={Jz:.2f}π')
    if Lx == 0:
        axs[L-Lx-1, Lz].set_xlabel('Time')
    if Lz == 0:
        axs[L-Lx-1, Lz].set_ylabel('Magnetization')
    axs[L-Lx-1, Lz].set_xscale('log')
    axs[L-Lx-1, Lz].set_yscale('log')
    axs[L-Lx-1, Lz].legend()
fig.suptitle(f'Phase Space Diagram U(1) with N={N}, #cr={circuit_realizations}', fontsize=16)
fig.tight_layout()
#set title to the whole figure

In [51]:
fn.print_matrix(fn.gate_xyz_disordered(0,0,np.pi/8, np.pi/8, np.pi/8, 0,0))

(0.9239-0.3827j)	.               	.               	.               
.               	(0.6533+0.2706j)	(0.2706-0.6533j)	.               
.               	(0.2706-0.6533j)	(0.6533+0.2706j)	.               
.               	.               	.               	(0.9239-0.3827j)


### Check that gates match

In [ ]:
import functions as fn
import numpy as np
sx, sy, sz, id_ = fn.X, fn.Y, fn.Z, fn.I

def print_paulis(gave_evo):
    PAULI = {'Z': sz, 'Y': sy, '1': id_, 'X': sx, }
    for i in PAULI:
        pi = PAULI[i]
        for j in PAULI:
            pj = PAULI[j]; overlap = np.trace(gave_evo @ np.kron(pi, pj))
            if np.abs(overlap) > 1e-10:
                print(i,j,f' -> {overlap.real:.9f}', sep='')

In [ ]:
with open('../random_archive/angles_N2.json', 'r', encoding='utf-8') as f:
    loaded_dict = json.load(f)
params = loaded_dict[0][0]
Jz = params['Jz']; Jx = params['Jx']
θ1 = params['θ1']; θ2 = params['θ2']
θ3 = params['θ3']; θ4 = params['θ4']

ale = fn.gate_xyz_disordered(θ1, θ2, Jx/2, -Jx/2, Jz/2, θ3, θ4)
fn.print_matrix(ale)

gate = np.kron(sz, sx)/4
print(f'\n','~'*30,'\nbefore evolution:\n', sep='')
fn.print_matrix(gate, 2)
print_paulis(gate)
gave_evo = ale.conj().T @ gate @ ale
print(f'\n','~'*30,'\nafter evolution:\n', sep='')
fn.print_matrix(gave_evo, 2)
print_paulis(gave_evo)